In [5]:
import subprocess
import socket
from flask import Flask

app = Flask(__name__)

def get_local_ip():
    s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    s.connect(("8.8.8.8", 80))
    ip = s.getsockname()[0]
    s.close()
    return ip


def setup(ip):
    subprocess.run(["server", ip])  # optional external script


def start_server():
    app.run(host="0.0.0.0", port=8080)


def remote_browser(ip):
    confirm = input(f"{ip} is correct? [y/n]: ").strip().lower()

    if confirm != "y":
        print("Cancelled")
        return

    setup(ip)
    start_server()


# --- MAIN ---
local_ip = get_local_ip()
local_ip = '127.0.0.1'
ip_access_url = f"http://{local_ip}:8080"

print("Server will be available at:")
print(ip_access_url)

remote_browser(local_ip)

Server will be available at:
http://127.0.0.1:8080


127.0.0.1 is correct? [y/n]:  


Cancelled


In [1]:
import os
import socket
import logging
import subprocess
import requests
from flask import Flask, request, jsonify, send_from_directory

log = logging.getLogger(__name__)

STATIC_DIR = os.path.join(os.path.dirname(__file__), "static")
app = Flask(__name__, static_folder=STATIC_DIR, static_url_path="/static")

DEFAULT_PORT    = 8080
DEFAULT_TIMEOUT = 10

# ---------------------------------------------------------------------------
# Frontend routes
# ---------------------------------------------------------------------------

@app.route("/")
def index():
    """Serve the frontend."""
    return send_from_directory(STATIC_DIR, "index.html")

# ---------------------------------------------------------------------------
# API routes
# ---------------------------------------------------------------------------

@app.route("/landing", methods=["POST"])
def landing():
    """Receive handshake from send_link."""
    data = request.get_json(silent=True)
    if not data or "key_hash" not in data:
        return jsonify({"error": "missing key_hash"}), 400
    log.info(f"Landing zone hit — key_hash: {data['key_hash']}")
    return jsonify({"status": "ready"}), 200

@app.route("/receive", methods=["POST"])
def receive():
    """Receive encrypted credential payload."""
    data = request.get_json(silent=True)
    if not data or "encrypted" not in data:
        return jsonify({"error": "missing encrypted payload"}), 400
    log.info("Credential received.")
    # todo: pass to validate_credentials + store_credentials
    return jsonify({"status": "received"}), 200

@app.route("/status", methods=["GET"])
def status():
    """Health check endpoint polled by the frontend."""
    return jsonify({"status": "online", "ip": get_local_ip()}), 200

# ---------------------------------------------------------------------------
# Transmit (client side)
# ---------------------------------------------------------------------------

def transmit(client_pkg: dict) -> bool:
    endpoint  = client_pkg.get("endpoint")
    encrypted = client_pkg.get("key")

    if not endpoint or not encrypted:
        log.error("transmit: missing 'endpoint' or 'key'")
        return False

    payload = {
        "encrypted": encrypted.hex() if isinstance(encrypted, bytes) else encrypted
    }

    try:
        response = requests.post(
            f"{endpoint}/receive",
            json=payload,
            timeout=DEFAULT_TIMEOUT,
        )
        response.raise_for_status()
        log.info(f"Credential transmitted — status {response.status_code}")
        return True

    except requests.exceptions.ConnectionError:
        log.error(f"transmit: could not reach {endpoint}")
    except requests.exceptions.Timeout:
        log.error(f"transmit: timed out after {DEFAULT_TIMEOUT}s")
    except requests.exceptions.HTTPError as e:
        log.error(f"transmit: server error: {e}")

    return False

# ---------------------------------------------------------------------------
# Server setup
# ---------------------------------------------------------------------------

def get_local_ip() -> str:
    s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    try:
        s.connect(("8.8.8.8", 80))
        return s.getsockname()[0]
    finally:
        s.close()

def setup(ip: str) -> None:
    script = os.environ.get("SETUP_SCRIPT")
    if script:
        subprocess.run([script, ip], check=True)
    else:
        log.info("No SETUP_SCRIPT env var set — skipping external setup")

def start_server(host: str = "0.0.0.0", port: int = DEFAULT_PORT) -> None:
    log.info(f"Starting server on {host}:{port}")
    app.run(host=host, port=port)

def remote_browser(ip: str) -> None:
    confirm = input(f"Start server at {ip}:{DEFAULT_PORT}? [y/n]: ").strip().lower()
    if confirm != "y":
        print("Cancelled.")
        return
    setup(ip)
    start_server()

# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    local_ip = get_local_ip()
    print(f"Server will be available at: http://{local_ip}:{DEFAULT_PORT}")
    remote_browser(local_ip)

NameError: name '__file__' is not defined

In [11]:

# requires port 22 be open or explicit permit rule inserted to IP/PF/NF (tables, config)
# sudo iptables -I INPUT -p tcp -s 192.168.1.50 --dport 8080 -j ACCEPT
# docker run -p 192.168.1.50:8080:80 myimage
# sudo iptables -I INPUT -p tcp --dport 8080 -j DROP
# mac enable default pf?
import subprocess

def ssh(username, ip):
    result = subprocess.run(
        [
            "ssh", 
            f"{username}@{ip}", 
            "bash ./ssh-script.sh"
        ], 
        check=True, 
        capture_output=True, 
        text=True 
    )
    print(result.stdout)
    print(result.stderr)

# uname = subprocess.run('whoami', capture_output=True, text=True ).stdout.strip()
# ip = subprocess.run(['ipconfig', 'getifaddr', 'en0'], capture_output=True, text=True ).stdout.strip()

# ssh(uname, ip)

In [ ]:
import subprocess
import logging

log = logging.getLogger(__name__)

DEFAULT_SCRIPT  = "bash ./ssh-script.sh"
DEFAULT_TIMEOUT = 30  # seconds


def ssh(
    username: str,
    ip: str,
    script: str  = DEFAULT_SCRIPT,
    port: int    = 22,
    key_path: str = None,
) -> bool:
    """
    Open an SSH connection and execute a remote script.

    Args:
        username:  remote user
        ip:        target IP or hostname
        script:    command to run on remote (default: bash ./ssh-script.sh)
        port:      SSH port (default: 22)
        key_path:  path to private key file — uses ssh-agent if None

    Returns:
        True on success, False on failure

    Requirements:
        - Port 22 open on target, or explicit permit rule:
            sudo iptables -I INPUT -p tcp -s <your_ip> --dport 22 -j ACCEPT
        - Remote script ./ssh-script.sh must exist on target machine

    macOS pf note:
        To allow outbound SSH ensure pf is not blocking port 22:
            sudo pfctl -f /etc/pf.conf
    """
    cmd = ["ssh", "-p", str(port)]

    if key_path:
        cmd += ["-i", key_path]

    cmd += [
        "-o", "StrictHostKeyChecking=accept-new",  # auto-accept new host keys
        "-o", f"ConnectTimeout={DEFAULT_TIMEOUT}",
        f"{username}@{ip}",
        script,
    ]

    log.info(f"SSH → {username}@{ip}:{port} running: '{script}'")

    try:
        result = subprocess.run(
            cmd,
            check=True,
            capture_output=True,
            text=True,
            timeout=DEFAULT_TIMEOUT,
        )
        if result.stdout:
            log.info(f"stdout:\n{result.stdout.strip()}")
        if result.stderr:
            log.warning(f"stderr:\n{result.stderr.strip()}")
        return True

    except subprocess.CalledProcessError as e:
        log.error(f"SSH command failed (exit {e.returncode}):\n{e.stderr.strip()}")
    except subprocess.TimeoutExpired:
        log.error(f"SSH timed out after {DEFAULT_TIMEOUT}s")
    except FileNotFoundError:
        log.error("'ssh' binary not found — is OpenSSH installed?")

    return False


def get_local_username() -> str:
    """Return current OS username."""
    result = subprocess.run("whoami", capture_output=True, text=True)
    return result.stdout.strip()


def get_local_ip(interface: str = "en0") -> str:
    """Return IP of the given network interface (macOS default: en0)."""
    result = subprocess.run(
        ["ipconfig", "getifaddr", interface],
        capture_output=True,
        text=True,
    )
    return result.stdout.strip()

In [12]:
import subprocess
# MacOS
# uname = subprocess.run('whoami', capture_output=True, text=True ).stdout.strip()
# ip = subprocess.run(['ipconfig', 'getifaddr', 'en0'], capture_output=True, text=True ).stdout.strip()
uname = 'test-user'
ip = '127.0.0.1'
print(f'{uname}@{ip}')

test-user@127.0.0.1


In [15]:
import subprocess

def local_info():
    hostname = subprocess.run(
        ["hostname"],
        capture_output=True,
        text=True
    ).stdout.strip()

    ip = subprocess.run(
        ["ip", "route", "get", "1.1.1.1"],
        capture_output=True,
        text=True
    ).stdout.split("src")[-1].strip().split()[0]

    return hostname, ip
# print(local_info())
# ssh(local_info()[0], local_info()[1])